In [1]:
import cv2
import yolov5
import easyocr
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# load model
model = yolov5.load('./model.pt') 

In [ ]:
reader = easyocr.Reader(["en"])

In [4]:
# set model parameters
model.conf = 0.25  # NMS confidence threshold
model.iou = 0.45  # NMS IoU threshold
model.agnostic = False  # NMS class-agnostic
model.multi_label = False  # NMS multiple labels per box
model.max_det = 1000  # maximum number of detections per image

In [32]:
# set image path
img = './test_videos/test_pic1.jpg'

In [89]:
def load_img(path):
    # the_img = cv2.imread(path)
    # cv2.resize(the_img, (120, 120))
    return cv2.imread(path)

def extract_text_from_image(image):
    # reader = easyocr.Reader(['en'])  # Language(s) for text recognition
    # image = cv2.imread(image_path)
    result = reader.readtext(image)
    texts = [entry[1] for entry in result]  # Extract text from result
    return texts
        
def process_image(img_path):
    original_img = load_img(img_path)
    results = model(original_img, augment=True)
    predictions = results.pred[0]
    car_plate_results = predictions[:, :4].tolist()  # Get bounding box information
    for i in range(len(car_plate_results)):
        xmin, ymin, xmax, ymax = map(int, car_plate_results[i])
        cropped_image = cv2.cvtColor(original_img[ymin:ymax, xmin:xmax], cv2.COLOR_BGR2RGB)
        plate_text = extract_text_from_image(cropped_image)
        print('Plate Text:', plate_text)

In [90]:
process_image(img)

Plate Text: ['21245 CUL']
Plate Text: ['T945', 'DTS']


#### Everything is done until here. Below is just some tries

In [72]:
image = load_img(img)
results = model(image, augment=True)

In [24]:
def plate_detection(path: str):
    # Read image
    image = load_img(path)
    
    # perform inference
    # results = model(img)

    # inference with larger input size
    # results = model(img, size=640)

    # inference with test time augmentation
    results = model(image, augment=True)

    # parse results
    print(results.pred)
    predictions = results.pred[0]
    boxes = predictions[:, :4] # x1, y1, x2, y2
    scores = predictions[:, 4]
    categories = predictions[:, 5]
    
    # getting bounding box coordinates
    xmin, ymin, xmax, ymax = map(int, boxes.tolist()[0])
    pt1 =(xmin,ymin)
    pt2 =(xmax,ymax)

    # cropped_image = image[ymin:ymax, xmin:xmax]
    cropped_image = cv2.cvtColor(image[ymin:ymax, xmin:xmax], cv2.COLOR_BGR2RGB)
    
    cv2.rectangle(image, pt1, pt2, (0, 0, 255), 10)
    cv2.putText(image, "Licence", pt1, cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 255), 10)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cropped_image, image, boxes

In [ ]:
cropped, image, cods = plate_detection(img)
plt.imshow(image)

### Commented out this cell
`Its Purpose was to obtain a good grayscaling for an easy way for OCR to extract text.`

`Found None out of them`

```python
gray_cropped_image = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
ret,thresh1 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_BINARY)
ret,thresh2 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_BINARY_INV)
ret,thresh3 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TRUNC)
ret,thresh4 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TOZERO)
ret,thresh5 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TOZERO_INV)

titles = ['Original Image','BINARY with OTSU','BINARY_INV','TRUNC from OTSU','TOZERO','TOZERO_INV']
images = [cropped, thresh1, thresh2, thresh3, thresh4, thresh5]

for i in range(6):
    plt.subplot(2,3,i+1),plt.imshow(images[i],'gray',vmin=0,vmax=255)
    plt.title(titles[i])
    plt.xticks([]),plt.yticks([])
```

In [12]:
results = reader.readtext(cropped)

In [ ]:
for (bbox, text, prob) in results:
    # Extract coordinates
    bbox = np.round(bbox).astype(np.int32)
    # print(bbox, text)
    top_left = tuple(bbox[0])
    bottom_right = tuple(bbox[2])
    # print(top_left, bottom_right)

    # Draw bounding box and text
    cv2.rectangle(cropped, top_left, bottom_right, (0, 255, 0), 2)
    # cv2.putText(cropped_image, text, top_left, cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

results

In [ ]:
plt.imshow(cropped)

# Key Take Aways
- The algorithm, can Detect the plate number in multiple orientations
- Detecting plates on multiple orientations means It can even start to extract plate letters then numbers

In [14]:
# save results into "results/" folder
# results.save(save_dir='results/')